<a href="https://colab.research.google.com/github/ennergarcia/Biblioteca_Python_Pandera-Validacao_Dados/blob/main/exemplo2_pandera.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## <font color='white'>Exemplo Prático 2 - Linguagem Python no Jupyter Notebook</font>

Problema:

Imagine que você tem um DataFrame de vendas e com o Pandera necessite validar as vendas por estado controlando os dados para evitar itens como: "São Paulo", "SP", "S. Paulo".

In [5]:
# Realizando os imports
!pip install pandera
import pandas as pd
import pandera.pandas as pa
from pandera import Column, Check

In [6]:
# 1. Definimos a lista de UFs permitidas (padrão IBGE)
UFS_BRASIL = [
    'AC', 'AL', 'AP', 'AM', 'BA', 'CE', 'DF', 'ES', 'GO',
    'MA', 'MT', 'MS', 'MG', 'PA', 'PB', 'PR', 'PE', 'PI',
    'RJ', 'RN', 'RS', 'RO', 'RR', 'SC', 'SP', 'SE', 'TO'
]

In [7]:
# 2. Criamos o Esquema de Validação
schema_vendas = pa.DataFrameSchema({
    "id_venda": Column(int, unique=True),
    "valor_venda": Column(float, Check.greater_than_or_equal_to(0)),

    # Validação da UF: deve ser string e estar dentro da nossa lista oficial
    "estado_uf": Column(
        str,
        Check.isin(UFS_BRASIL),
        description="Sigla da UF com dois caracteres maiúsculos"
    )
})

In [8]:
# 3. Exemplo de DataFrame com erro (um estado por extenso)
df_vendas = pd.DataFrame({
    "id_venda": [101, 102, 103],
    "valor_venda": [250.50, 1200.00, 89.90],
    "estado_uf": ["SP", "Rio de Janeiro", "MG"]  # "Rio de Janeiro" causará erro!
})

In [9]:
# 4. Execução da Validação
try:
    schema_vendas.validate(df_vendas, lazy=True)
except pa.errors.SchemaErrors as err:
    print("❌ Falha na Validação de Dados!")
    # O parâmetro lazy=True permite capturar todos os erros de uma vez
    print(err.failure_cases[['column', 'index', 'failure_case']])

❌ Falha na Validação de Dados!
      column  index    failure_case
0  estado_uf      1  Rio de Janeiro


## <font color='white'>Erro encontrado!!!</font>

Conforme a mensagem foi encontrado um valor "Rio de Janeiro" fora dos parâmetros configurados.